In [74]:
using LowLevelFEM, LinearAlgebra

In [75]:
openGeometry("box.geo")

In [76]:
#openPreProcessor()

In [77]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [78]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 2000)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(Symmetric(K), f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

  0.295912 seconds (169.14 k allocations: 23.556 MiB, 2.86% gc time, 35.03% compilation time)


0

In [79]:
C = contact(u, master="master", slave="slave", cn=1e8)

Contact("slave" -> "master", 303 candidate nodes, 152 active, stick=152, slip=0, G=(909, 4269), C=(909, 909))

In [80]:
pdim = C.U.pdim

support = [bc_bottom, bc_top]
free = freeDoFs(U, support)

u_it = copy(u)

old_tags = copy(C.master_element_tags)
old_G = copy(C.G)

for iter in 1:30

    updateContact!(C, u_it)

    nchanged = count(old_tags .!= C.master_element_tags)

    dG = norm(C.G - old_G) /
        max(norm(old_G), eps())

    println(
        "master changes = ", nchanged,
        ", dG = ", dG
    )

    old_tags = copy(C.master_element_tags)
    old_G = copy(C.G)

    gc = zeros(size(C.G, 1))

    for i in eachindex(C.slave_nodes)
        if C.active[i]
            row_n = (i - 1) * pdim + 1
            gc[row_n] = C.gap_values[i]
        end
    end

    # Contact residual and tangent
    rc = C.G' * (C.C * gc)
    Kc = C.G' * C.C * C.G

    # Total residual
    r = K.A * u_it.a[:,1] - f.a[:,1] + rc

    # Current tangent
    A = K.A + Kc

    # Homogeneous correction on prescribed DoFs
    Δu = zeros(length(r))
    Δu[free] = -A[free, free] \ r[free]

    r0 = norm(r[free])

    α = 1.0
    u_trial = copy(u_it)

    while α > 1e-6
        u_trial.a[:, 1] .= u_it.a[:, 1] .+ α .* Δu

        updateContact!(C, u_trial)

        gc_trial = zeros(size(C.G, 1))
        for i in eachindex(C.slave_nodes)
            if C.active[i]
                row_n = (i - 1) * pdim + 1
                gc_trial[row_n] = C.gap_values[i]
            end
        end

        rc_trial = C.G' * (C.C * gc_trial)

        r_trial =
            K.A * u_trial.a[:, 1] -
            f.a[:, 1] +
            rc_trial

        if norm(r_trial[free]) < r0
            break
        end

        α *= 0.5
    end

    u_it.a[:, 1] .= u_trial.a[:, 1]

    err = α * norm(Δu[free]) /
          max(norm(u_it.a[free,1]), eps())

    println(
        "iter = ", iter,
        ", α = ", α,
        ", active = ", count(C.active),
        ", min gap = ", minimum(C.gap_values),
        ", error = ", err,
        ", |r| = ", norm(r[free])
    )

    err < 1e-8 && break
end

u = u_it

master changes = 0, dG = 8.726151675064791e-17
master switch: 30 -> 32, d_old = 0.11857415871614878, d_new = 0.11857415871614872, Δd = 5.551115123125783e-17, ξ_old = (0.0, 1.0), ξ_new = (0.0, 0.0)
master switch: 135 -> 136, d_old = 0.0016649775257564657, d_new = 8.700704729695112e-5, Δd = 0.0015779704784595146, ξ_old = (0.8912481683812731, 0.10875183161872692), ξ_new = (0.0019125224101928517, 0.10779574255964122)
master switch: 3 -> 4, d_old = 0.09528290095388692, d_new = 0.09528196048666558, Δd = 9.404672213358944e-7, ξ_old = (0.5972193277033534, 0.4027806722966466), ξ_new = (0.5967142875975322, 0.0005780905988568286)
iter = 1, α = 1.0, active = 91, min gap = -9.965718556745859e-5, error = 0.23159038855395483, |r| = 2.0726951335736942e8
master changes = 3, dG = 0.07333275870319386
master switch: 136 -> 135, d_old = 0.002087776989998409, d_new = 3.972362181941168e-5, Δd = 0.0020480533681789975, ξ_old = (0.0, 0.1111350029513729), ξ_new = (0.888860155388928, 0.10904871224377653)
master s

nodal VectorField
[0.0; 0.0; … ; -0.1090109443392662; 0.023456716230795583;;]

In [81]:
showDoFResults(u, name="u cont.", visible=true, factor=1)

1

In [85]:
showElementResults(C.gap, name="gap")

3

In [86]:
openPostProcessor()